# ASL Sign Language Recognition - End-to-End Kaggle Pipeline

Notebook all-in-one phục vụ training mô hình nhận diện ASL Sign Language trên **Kaggle Notebook**.

### Quy trình tích hợp:
1. **Environment Setup**: Kiểm tra môi trường Kaggle, GPU (CUDA), PyTorch.
2. **Clone Repo GitHub**: Clone mã nguồn dự án vào `/kaggle/working` và cài đặt dependencies (`mediapipe`).
3. **Dataset Discovery & Loading**: Tự động nhận diện dataset `sttaseen/wlasl2000-resized` từ `/kaggle/input`.
4. **Class Selection & Splits**: Phân tích metadata `WLASL_v0.3.json`, chọn 5 classes khởi đầu và giữ nguyên official train/val/test splits.
5. **Feature Extraction**: Trích xuất MediaPipe Hand landmarks (2 tay × 21 điểm × 3 tọa độ = 126 đặc trưng/frame).
6. **Preprocessing & DataLoader**: Chuẩn hóa không gian và resampling cố định chiều dài chuỗi ($T=32$).
7. **Model Architecture**: Định nghĩa mô hình PyTorch GRU Classifier.
8. **Training Loop**: Huấn luyện với early saving cho checkpoint tốt nhất (`best_model.pth`).
9. **Evaluation**: Đánh giá trên tập test, vẽ biểu đồ Loss/Accuracy và Confusion Matrix.

--- 
## 1. Setup môi trường & Kiểm tra GPU

In [ ]:
import os
import sys
from pathlib import Path
import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("WARNING: GPU is not enabled. Please enable GPU accelerator in Kaggle Settings (Right sidebar).")
    device = torch.device("cpu")


--- 
## 2. Clone GitHub Repository & Cài đặt Dependencies

Clone repository vào `/kaggle/working/SignLanguageReconition_ML` và thêm vào `sys.path`.

In [ ]:
# Cài đặt mediapipe (Kaggle đã có sẵn torch, opencv, pandas, sklearn...)
!pip install -q mediapipe

REPO_URL = "https://github.com/ChienPM-27/SignLanguageReconition_ML.git"
BRANCH = "main"  # Nhánh chính chứa bản cập nhật mới nhất (hoặc đổi thành "develop" nếu cần)
TARGET_DIR = Path("/kaggle/working/SignLanguageReconition_ML")

if not TARGET_DIR.exists():
    print(f"Cloning {REPO_URL} (branch: {BRANCH}) into {TARGET_DIR}...")
    !git clone -b {BRANCH} {REPO_URL} {TARGET_DIR}
else:
    print(f"Repo already exists at {TARGET_DIR}. Pulling latest changes from {BRANCH}...")
    !cd {TARGET_DIR} && git checkout {BRANCH} && git pull

if str(TARGET_DIR) not in sys.path:
    sys.path.insert(0, str(TARGET_DIR))
print("Repo ready and added to sys.path!")


--- 
## 3. Tự động nhận diện Dataset trong `/kaggle/input`

Tìm thư mục dataset `WLASL-2000 Resized` được gắn trên Kaggle (`sttaseen/wlasl2000-resized`).

In [ ]:
kaggle_input = Path("/kaggle/input")

# Tìm kiếm thư mục chứa dataset WLASL
dataset_dirs = [d for d in kaggle_input.iterdir() if d.is_dir() and "wlasl" in d.name.lower()]

if dataset_dirs:
    DATA_ROOT = dataset_dirs[0]
    print(f"Found WLASL Dataset Directory: {DATA_ROOT}")
else:
    # Fallback mặc định theo slug của Kaggle
    DATA_ROOT = kaggle_input / "wlasl2000-resized"
    print(f"Defaulting DATA_ROOT to: {DATA_ROOT}")

print("\n--- Files & Directories inside DATA_ROOT ---")
if DATA_ROOT.exists():
    for item in DATA_ROOT.iterdir():
        print(f" - {item.name} ({'Dir' if item.is_dir() else 'File'})")
else:
    print(f"[ERROR] DATA_ROOT '{DATA_ROOT}' does not exist!")
    print("Vui lòng bấm '+ Add Input' trên Kaggle và thêm dataset: sttaseen/wlasl2000-resized")

# Thiết lập thư mục output ghi được trong /kaggle/working
OUTPUT_ROOT = Path("/kaggle/working/asl_outputs")
LANDMARK_ROOT = OUTPUT_ROOT / "landmarks"
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
PLOTS_ROOT = OUTPUT_ROOT / "plots"

for d in [OUTPUT_ROOT, LANDMARK_ROOT, CHECKPOINT_ROOT, PLOTS_ROOT]:
    d.mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory ready: {OUTPUT_ROOT}")


--- 
## 4. Đọc Metadata & Chọn 5 Classes Khởi Đầu

Trích xuất annotations từ `WLASL_v0.3.json`, kiểm tra official splits (`train`, `val`, `test`) và signer ID.

In [ ]:
import json
import pandas as pd

# Xác định file JSON metadata
json_candidates = list(DATA_ROOT.glob("*.json"))
print(f"Candidate JSON files found: {[f.name for f in json_candidates]}")

wlasl_json = None
for f in json_candidates:
    if "v0.3" in f.name.lower() or "wlasl" in f.name.lower():
        wlasl_json = f
        break
if not wlasl_json and json_candidates:
    wlasl_json = json_candidates[0]

print(f"Using Metadata JSON: {wlasl_json}")

with open(wlasl_json, "r", encoding="utf-8") as f:
    raw_meta = json.load(f)

# Phẳng hóa danh sách instances
all_records = []
for entry in raw_meta:
    gloss = entry.get("gloss")
    for inst in entry.get("instances", []):
        record = dict(inst)
        record["gloss"] = gloss
        all_records.append(record)

df_meta = pd.DataFrame(all_records)
print(f"Total dataset instances: {len(df_meta)}")
print(f"Total unique glosses: {df_meta['gloss'].nunique()}")

# Chọn 5 classes có nhiều video nhất làm tập khởi đầu
NUM_CLASSES = 5
top_classes = df_meta["gloss"].value_counts().head(NUM_CLASSES).index.tolist()
print(f"\nTop {NUM_CLASSES} Selected Classes: {top_classes}")

# Lọc dữ liệu theo 5 classes
df_selected = df_meta[df_meta["gloss"].isin(top_classes)].copy()
print("\nDistribution across official splits:")
print(df_selected["split"].value_counts())

# Kiểm tra phân bố Signers để tránh data leakage
train_signers = set(df_selected[df_selected["split"] == "train"]["signer_id"].dropna())
val_signers = set(df_selected[df_selected["split"] == "val"]["signer_id"].dropna())
test_signers = set(df_selected[df_selected["split"] == "test"]["signer_id"].dropna())
print(f"Signer Overlap Train/Val: {len(train_signers.intersection(val_signers))}")
print(f"Signer Overlap Train/Test: {len(train_signers.intersection(test_signers))}")


--- 
## 5. Trích xuất MediaPipe Hand Landmarks

Mỗi frame trích xuất:
- 2 bàn tay (Trái + Phải)
- 21 keypoints/tay × 3 tọa độ (x, y, z) = **126 features/frame**.
- Nếu chỉ có 1 tay hoặc không phát hiện tay, phần còn lại được zero-pad.

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from tqdm.auto import tqdm

# Xác định thư mục chứa file video .mp4
video_dirs = list(DATA_ROOT.glob("**/videos")) + list(DATA_ROOT.glob("**/*video*"))
if video_dirs:
    VIDEO_DIR = video_dirs[0]
else:
    VIDEO_DIR = DATA_ROOT
print(f"Videos Directory: {VIDEO_DIR}")

class MediaPipeExtractor:
    def __init__(self):
        self.mp_hands = mp.solutions.hands
        self.hands = self.mp_hands.Hands(
            static_image_mode=False,
            max_num_hands=2,
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

    def extract_from_frame(self, frame: np.ndarray) -> np.ndarray:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = self.hands.process(frame_rgb)
        
        left_hand = np.zeros((21, 3), dtype=np.float32)
        right_hand = np.zeros((21, 3), dtype=np.float32)
        
        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                label = handedness.classification[0].label
                coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark], dtype=np.float32)
                if label == "Left":
                    left_hand = coords
                elif label == "Right":
                    right_hand = coords
        return np.concatenate([left_hand.flatten(), right_hand.flatten()], axis=0)

    def extract_from_video(self, video_path: Path) -> np.ndarray:
        cap = cv2.VideoCapture(str(video_path))
        sequence = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            feat = self.extract_from_frame(frame)
            sequence.append(feat)
        cap.release()
        if len(sequence) == 0:
            return np.zeros((0, 126), dtype=np.float32)
        return np.array(sequence, dtype=np.float32)

    def close(self):
        self.hands.close()

extractor = MediaPipeExtractor()

# Trích xuất landmark cho các video đã chọn và lưu cache .npy
valid_samples = []
print(f"Processing landmarks for {len(df_selected)} video samples...")

for idx, row in tqdm(df_selected.iterrows(), total=len(df_selected)):
    vid_id = row["video_id"]
    gloss = row["gloss"]
    split = row["split"]
    
    # Tìm file video trên disk
    vid_path = VIDEO_DIR / f"{vid_id}.mp4"
    if not vid_path.exists():
        # Thử tìm đệ quy nếu tên file có dạng khác
        matches = list(DATA_ROOT.glob(f"**/{vid_id}.mp4"))
        if matches:
            vid_path = matches[0]
        else:
            continue  # Bỏ qua nếu thiếu video
            
    out_npy = LANDMARK_ROOT / f"{vid_id}.npy"
    if not out_npy.exists():
        landmarks = extractor.extract_from_video(vid_path)
        if len(landmarks) > 0:
            np.save(out_npy, landmarks)
            valid_samples.append({"video_id": vid_id, "gloss": gloss, "split": split, "npy_path": str(out_npy)})
    else:
        valid_samples.append({"video_id": vid_id, "gloss": gloss, "split": split, "npy_path": str(out_npy)})

extractor.close()
df_valid = pd.DataFrame(valid_samples)
print(f"Successfully extracted landmarks for {len(df_valid)} videos.")


--- 
## 6. Preprocessing & PyTorch Dataset/DataLoader

- **Chuẩn hóa không gian**: Đưa cổ tay về gốc tọa độ và scale theo kích thước bàn tay.
- **Chuẩn hóa thời gian**: Resampling chuỗi thời gian về độ dài cố định $T = 32$ frames.

In [ ]:
from torch.utils.data import Dataset, DataLoader

def normalize_hand(hand_coords: np.ndarray) -> np.ndarray:
    if np.all(hand_coords == 0):
        return hand_coords
    wrist = hand_coords[0:1, :]
    translated = hand_coords - wrist
    scale = np.linalg.norm(translated[9] - translated[0])
    return translated / (scale if scale > 1e-6 else 1.0)

def normalize_frame(features: np.ndarray) -> np.ndarray:
    left = normalize_hand(features[:63].reshape(21, 3)).flatten()
    right = normalize_hand(features[63:].reshape(21, 3)).flatten()
    return np.concatenate([left, right], axis=0)

def resample_sequence(sequence: np.ndarray, target_len: int = 32) -> np.ndarray:
    t = len(sequence)
    if t == 0:
        return np.zeros((target_len, 126), dtype=np.float32)
    if t == target_len:
        return sequence
    indices = np.linspace(0, t - 1, target_len)
    resampled = np.zeros((target_len, sequence.shape[1]), dtype=np.float32)
    for i in range(sequence.shape[1]):
        resampled[:, i] = np.interp(indices, np.arange(t), sequence[:, i])
    return resampled

class ASLLandmarkDataset(Dataset):
    def __init__(self, df: pd.DataFrame, class_to_idx: dict, sequence_length: int = 32):
        self.samples = df.to_dict("records")
        self.class_to_idx = class_to_idx
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        raw_seq = np.load(item["npy_path"])
        # Chuẩn hóa không gian từng frame
        norm_seq = np.array([normalize_frame(f) for f in raw_seq], dtype=np.float32)
        # Resample về fixed sequence length
        fixed_seq = resample_sequence(norm_seq, target_len=self.sequence_length)
        label = self.class_to_idx[item["gloss"]]
        return torch.tensor(fixed_seq, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

# Mapping class name -> int
class_to_idx = {cls: idx for idx, cls in enumerate(top_classes)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}
print("Class Mapping:", class_to_idx)

df_train = df_valid[df_valid["split"] == "train"]
df_val = df_valid[df_valid["split"] == "val"]
df_test = df_valid[df_valid["split"] == "test"]

print(f"Samples -> Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

BATCH_SIZE = 32
train_loader = DataLoader(ASLLandmarkDataset(df_train, class_to_idx), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(ASLLandmarkDataset(df_val, class_to_idx), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(ASLLandmarkDataset(df_test, class_to_idx), batch_size=BATCH_SIZE, shuffle=False)


--- 
## 7. Kiến trúc mô hình PyTorch GRU

In [ ]:
import torch.nn as nn

class ASLGRUClassifier(nn.Module):
    def __init__(
        self,
        input_size: int = 126,
        hidden_size: int = 128,
        num_layers: int = 2,
        num_classes: int = 5,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, 32, 126]
        _, h_n = self.gru(x)
        # h_n[-1]: [B, hidden_size] (lớp GRU cuối cùng tại timestep cuối)
        last_hidden = h_n[-1]
        logits = self.classifier(last_hidden)
        return logits

model = ASLGRUClassifier(num_classes=len(top_classes)).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal Trainable Parameters: {total_params:,}")


--- 
## 8. Vòng lặp Huấn luyện (Training Loop) & Lưu Checkpoint

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

EPOCHS = 40
best_val_loss = float("inf")
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

print(f"Bắt đầu training trên {device} trong {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    # --- Train Phase ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        preds = model(x_b)
        loss = criterion(preds, y_b)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * x_b.size(0)
        train_correct += (preds.argmax(dim=1) == y_b).sum().item()
        train_total += y_b.size(0)
        
    epoch_train_loss = train_loss / max(train_total, 1)
    epoch_train_acc = train_correct / max(train_total, 1)
    
    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            preds = model(x_b)
            loss = criterion(preds, y_b)
            val_loss += loss.item() * x_b.size(0)
            val_correct += (preds.argmax(dim=1) == y_b).sum().item()
            val_total += y_b.size(0)
            
    epoch_val_loss = val_loss / max(val_total, 1)
    epoch_val_acc = val_correct / max(val_total, 1)
    
    scheduler.step(epoch_val_loss)
    
    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)
    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] Train Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.3f} | Val Loss: {epoch_val_loss:.4f} | Acc: {epoch_val_acc:.3f}")
        
    # Lưu best model
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), CHECKPOINT_ROOT / "best_model.pth")

# Lưu checkpoint cuối cùng
torch.save(model.state_dict(), CHECKPOINT_ROOT / "last_model.pth")
print(f"\nTraining hoàn tất! Checkpoint tốt nhất đã được lưu tại: {CHECKPOINT_ROOT / 'best_model.pth'}")


--- 
## 9. Đánh giá Mô hình trên Test Set & Trực quan hóa

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

# Load best checkpoint để đánh giá
model.load_state_dict(torch.load(CHECKPOINT_ROOT / "best_model.pth", map_location=device))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x_b, y_b in test_loader:
        x_b = x_b.to(device)
        preds = model(x_b).argmax(dim=1).cpu().numpy()
        y_pred.extend(preds)
        y_true.extend(y_b.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("=" * 60)
print("TEST SET CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=top_classes, zero_division=0))

# Vẽ biểu đồ Loss & Accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"], label="Val Loss")
axes[0].set_title("Training & Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history["train_acc"], label="Train Accuracy")
axes[1].plot(history["val_acc"], label="Val Accuracy")
axes[1].set_title("Training & Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.savefig(PLOTS_ROOT / "training_curves.png", bbox_inches="tight")
plt.show()

# Vẽ Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(len(top_classes)),
    yticks=np.arange(len(top_classes)),
    xticklabels=top_classes,
    yticklabels=top_classes,
    title="Test Set Confusion Matrix",
    ylabel="True Label",
    xlabel="Predicted Label"
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
for i in range(len(top_classes)):
    for j in range(len(top_classes)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.savefig(PLOTS_ROOT / "confusion_matrix.png", bbox_inches="tight")
plt.show()
